Exercise 1 — Three-qubit encoding

Requirements:
- Accept complex amplitudes.
- Validate normalization.
- Encode the state into \(\alpha|000\rangle+\beta|111\rangle\).
- Verify the output has norm 1.

Test at least these states:
- \(|0\rangle\)
- \(|1\rangle\)
- \(|+\rangle\)
- An arbitrary normalized superposition.

In [2]:
import numpy as np

I = np.eye(2, dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

Z = np.array([
    [1, 0],
    [0, -1]
], dtype=complex)

In [3]:
zero = np.array([1, 0], dtype=complex)
one = np.array([0, 1], dtype=complex)
plus = np.array([
    1 / np.sqrt(2),
    1 / np.sqrt(2)
], dtype=complex)


def encode_qubit (alpha, beta):
    # Return the three-qubit encoded state
    if round(np.abs(alpha)**2 + np.abs(beta)**2) != 1:
        raise ValueError("The coefficients alpha and beta must satisfy |alpha|^2 + |beta|^2 = 1.")

    ket000 = np.kron(np.kron(zero, zero), zero)
    ket111 = np.kron(np.kron(one, one), one)

    encoded_state = alpha * ket000 + beta * ket111
    print("Norm:", np.linalg.norm(encoded_state))
    
    return encoded_state

print("Encoded state:\n", encode_qubit(zero[0], zero[1]))
print("Encoded state:\n", encode_qubit(one[0], one[1]))
print("Encoded state:\n", encode_qubit(plus[0], plus[1]))
print("Encoded state:\n", encode_qubit(np.sqrt(3) / 2, 1 / 2))

Norm: 1.0
Encoded state:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Norm: 1.0
Encoded state:
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Norm: 0.9999999999999999
Encoded state:
 [0.70710678+0.j 0.        +0.j 0.        +0.j 0.        +0.j
 0.        +0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Norm: 0.9999999999999999
Encoded state:
 [0.8660254+0.j 0.       +0.j 0.       +0.j 0.       +0.j 0.       +0.j
 0.       +0.j 0.       +0.j 0.5      +0.j]


Exercise 2 — Simulate bit-flip errors

Construct the three bit-flip operators:

\[
X_1=X\otimes I\otimes I
\]

\[
X_2=I\otimes X\otimes I
\]

\[
X_3=I\otimes I\otimes X
\]

Apply each operator individually to your encoded state.

Verify that each error changes the encoded state.

In [4]:
X1 = np.kron(np.kron(X, I), I)
X2 = np.kron(np.kron(I, X), I)
X3 = np.kron(np.kron(I, I), X)

encoded_zero = encode_qubit(zero[0], zero[1])

bit_1_flip = X1 @ encoded_zero
bit_2_flip = X2 @ encoded_zero
bit_3_flip = X3 @ encoded_zero

print("Bit 1 flip:", not np.allclose(bit_1_flip, encoded_zero))
print("Bit 2 flip:", not np.allclose(bit_2_flip, encoded_zero))
print("Bit 3 flip:", not np.allclose(bit_3_flip, encoded_zero))

Norm: 1.0
Bit 1 flip: True
Bit 2 flip: True
Bit 3 flip: True


Exercise 3 — Syndrome detection

Construct:
\[
S_1=Z\otimes Z\otimes I
\]

\[
S_2=I\otimes Z\otimes Z
\]

Write a function that returns the two stabilizer measurement results.

Test the following:

- No error.
- Bit flip on qubit 1.
- Bit flip on qubit 2.
- Bit flip on qubit 3.

In [5]:
S1 = np.kron(np.kron(Z, Z), I)
S2 = np.kron(np.kron(I, Z), Z)

def stabilizer_measurement(state, operator):
    expectation = np.vdot(
        state,
        operator @ state
    )

    return np.real_if_close(expectation).item()

bit_flips = [encoded_zero, bit_1_flip, bit_2_flip, bit_3_flip]

for i, bit_flip in enumerate(bit_flips, start=0):
    s1 = stabilizer_measurement(bit_flip, S1)
    s2 = stabilizer_measurement(bit_flip, S2)

    if i == 0:
        print("No bit flip syndrome:", s1, s2)
    else:
        print(f"Bit {i} flip syndrome:", s1, s2)

No bit flip syndrome: 1.0 1.0
Bit 1 flip syndrome: -1.0 1.0
Bit 2 flip syndrome: -1.0 -1.0
Bit 3 flip syndrome: 1.0 -1.0


Exercise 4 — Automatic error correction

Implement a function that:

1. Receives an encoded state.
2. Measures its syndrome.
3. Determines the error location.
4. Applies the appropriate correction.
5. Returns the corrected state.

Test it against all three possible single-qubit bit flips.

In [6]:
def correct_bit_flip(state):

    s1 = round(stabilizer_measurement(state, S1))
    s2 = round(stabilizer_measurement(state, S2))

    syndrome = (s1, s2)

    if syndrome == (1, 1):
        corrected = state
        error = "No error"

    elif syndrome == (-1, 1):
        corrected = X1 @ state
        error = "Qubit 1 bit flip"

    elif syndrome == (-1, -1):
        corrected = X2 @ state
        error = "Qubit 2 bit flip"

    elif syndrome == (1, -1):
        corrected = X3 @ state
        error = "Qubit 3 bit flip"

    else:
        raise ValueError("Invalid syndrome")

    return corrected, error, syndrome

for i, bit_flip in enumerate(bit_flips, start=0):
    corrected_state, error, syndrome = correct_bit_flip(bit_flip)

    print(f"\nBit {i} flip correction:")
    print("Error:", error)
    print("Syndrome:", syndrome)
    print("Corrected state:\n", corrected_state)
    print("Norm:", np.linalg.norm(corrected_state))
    print("Corrected state matches original:", np.allclose(corrected_state, encoded_zero))


Bit 0 flip correction:
Error: No error
Syndrome: (1, 1)
Corrected state:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Norm: 1.0
Corrected state matches original: True

Bit 1 flip correction:
Error: Qubit 1 bit flip
Syndrome: (-1, 1)
Corrected state:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Norm: 1.0
Corrected state matches original: True

Bit 2 flip correction:
Error: Qubit 2 bit flip
Syndrome: (-1, -1)
Corrected state:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Norm: 1.0
Corrected state matches original: True

Bit 3 flip correction:
Error: Qubit 3 bit flip
Syndrome: (1, -1)
Corrected state:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Norm: 1.0
Corrected state matches original: True


Exercise 5 — Error correction under probabilistic noise

Instead of manually applying a particular bit flip, simulate a bit-flip channel independently on all three physical qubits.

For each qubit:
\[
P(\text{flip})=p
\]

Use Monte Carlo simulation with at least 10,000 trials.

Test:
\[
p\in\{0.01,0.05,0.10,0.20,0.30\}
\]

Calculate:

- Probability of successful recovery.
- Probability of an uncorrectable error.
- Compare the results with the theoretical success probability.

For independent bit-flip errors, the three-qubit repetition code succeeds when zero or one qubit flips:

\[
P_{\text{success}}
=
(1-p)^3+3p(1-p)^2
\]

The probability of failure is:

\[
P_{\text{failure}}
=
3p^2(1-p)+p^3
\]

Plot both probabilities against \(p\).

In [10]:
probabilities = [0.01, 0.05, 0.10, 0.20, 0.30]

for p in probabilities:
    bit_1_flip_chance = np.random.choice(
        [0, 1],
        size=10000,
        p=[1 - p, p]
    )

    bit_2_flip_chance = np.random.choice(
        [0, 1],
        size=10000,
        p=[1 - p, p]
    )

    bit_3_flip_chance = np.random.choice(
        [0, 1],
        size=10000,
        p=[1 - p, p]
    )

    bit_flips = []
    recoverable_count = 0
    for i in range(10000):
        state = encoded_zero.copy()
        bits_flipped = 0

        if bit_1_flip_chance[i] == 1:
            state = X1 @ state
            bits_flipped += 1

        if bit_2_flip_chance[i] == 1:
            state = X2 @ state
            bits_flipped += 1

        if bit_3_flip_chance[i] == 1:
            state = X3 @ state
            bits_flipped += 1

        bit_flips.append(state)

        if bits_flipped <= 1:
            recoverable_count += 1

    probability_recoverable = round(recoverable_count / 10000, 4)
    print("\n\nProbability of Recovery for bit flip probability", p, ":", probability_recoverable)
    print("Expected probability of recovery:", round((1 - p)**3 + 3 * p * (1 - p)**2, 4))

    print("\nProbability of Uncorrectable errors:", round(1 - probability_recoverable, 4))
    print("Expected probability of uncorrectable errors:", round(3 * p**2 * (1 - p) + p**3, 4))






Probability of Recovery for bit flip probability 0.01 : 0.9998
Expected probability of recovery: 0.9997

Probability of Uncorrectable errors: 0.0002
Expected probability of uncorrectable errors: 0.0003


Probability of Recovery for bit flip probability 0.05 : 0.9936
Expected probability of recovery: 0.9927

Probability of Uncorrectable errors: 0.0064
Expected probability of uncorrectable errors: 0.0073


Probability of Recovery for bit flip probability 0.1 : 0.9722
Expected probability of recovery: 0.972

Probability of Uncorrectable errors: 0.0278
Expected probability of uncorrectable errors: 0.028


Probability of Recovery for bit flip probability 0.2 : 0.8983
Expected probability of recovery: 0.896

Probability of Uncorrectable errors: 0.1017
Expected probability of uncorrectable errors: 0.104


Probability of Recovery for bit flip probability 0.3 : 0.7811
Expected probability of recovery: 0.784

Probability of Uncorrectable errors: 0.2189
Expected probability of uncorrectable err

Exercise 6 — Investigate phase-flip limitations

Use the same encoded state.

Apply a phase-flip operator to the first qubit:

\[
Z_1=Z\otimes I\otimes I
\]

Measure the two stabilizers.

Observe whether the syndrome changes.

In [12]:
Z1 = np.kron(np.kron(Z, I), I)

phase_1_flip = Z1 @ encoded_zero

s1 = round(stabilizer_measurement(phase_1_flip, S1))
s2 = round(stabilizer_measurement(phase_1_flip, S2))

syndrome = (s1, s2)

print("\nPhase flip syndrome:", syndrome)


Phase flip syndrome: (1, 1)
